[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/chain.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58238466-lesson-4-chain)

# Chain

## 回顧

我們建立了一個簡單的 graph，包含 node、normal edge 與 conditional edge。

## 目標

接下來，我們要建構一個簡單的 chain，結合 4 個概念。

* 使用 [chat messages](https://docs.langchain.com/oss/python/langchain/messages) 作為我們的 graph state
* 在 graph node 中使用 [chat models](https://docs.langchain.com/oss/python/integrations/chat)
* 為 chat model [綁定 tools](https://docs.langchain.com/oss/python/langchain/models#tool-calling)
* 在 graph node 中[執行 tool call](https://docs.langchain.com/oss/python/langchain/models#tool-execution-loop)

![Screenshot 2024-08-21 at 9.24.03 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbab08dd607b08df5e1101_chain1.png)

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langgraph

## Messages

Chat model 可以使用 [messages](https://docs.langchain.com/oss/python/langchain/messages)，這些 message 捕捉了對話中不同的角色。

LangChain 支援多種 message 型別，包括 `HumanMessage`、`AIMessage`、`SystemMessage` 與 `ToolMessage`。

它們分別代表：來自使用者的 message、來自 chat model 的 message、用來指示 chat model 行為的 message，以及來自 tool call 的 message。

讓我們建立一個 message 串列。

每個 message 都可以附帶以下幾項資訊：

* `content` —— message 的內容
* `name` —— message 的作者（選填）
* `response_metadata` —— 一個 metadata 的 dict（選填，例如常由 model provider 為 `AIMessages` 填入）

In [1]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage

messages = [AIMessage(content=f"So you said you were researching ocean mammals?", name="Model")]
messages.append(HumanMessage(content=f"Yes, that's right.",name="Lance"))
messages.append(AIMessage(content=f"Great, what would you like to learn about.", name="Model"))
messages.append(HumanMessage(content=f"I want to learn about the best place to see Orcas in the US.", name="Lance"))

for m in messages:
    print(m)

for m in messages:
    m.pretty_print()

content='So you said you were researching ocean mammals?' additional_kwargs={} response_metadata={} name='Model' tool_calls=[] invalid_tool_calls=[]
content="Yes, that's right." additional_kwargs={} response_metadata={} name='Lance'
content='Great, what would you like to learn about.' additional_kwargs={} response_metadata={} name='Model' tool_calls=[] invalid_tool_calls=[]
content='I want to learn about the best place to see Orcas in the US.' additional_kwargs={} response_metadata={} name='Lance'
================================== Ai Message ==================================
Name: Model

So you said you were researching ocean mammals?
================================ Human Message =================================
Name: Lance

Yes, that's right.
================================== Ai Message ==================================
Name: Model

Great, what would you like to learn about.
================================ Human Message =================================
Name: Lance

I want to l

## Chat Models

Chat model 以一連串 message 作為輸入，並支援上面討論過的各種 message 型別。

可選擇的[模型有很多](https://docs.langchain.com/oss/python/integrations/chat)！這裡我們使用 OpenAI。

讓我們確認你的 `OPENAI_API_KEY` 已設定好；如果沒有，系統會要求你輸入。

In [2]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

我們可以載入一個 chat model，並用我們的 message 串列來呼叫它。

可以看到結果是一個帶有特定 `response_metadata` 的 `AIMessage`。

In [2]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")
result = llm.invoke(messages)
type(result)

langchain_core.messages.ai.AIMessage

In [3]:
result

AIMessage(content='One of the best places to see orcas in the United States is the Pacific Northwest, particularly around the San Juan Islands in Washington State. Here are some details:\n\n1. **San Juan Islands, Washington**: These islands are a renowned spot for whale watching, with orcas frequently spotted between late spring and early fall. The waters around the San Juan Islands are home to both resident and transient orca pods, making it an excellent location for sightings.\n\n2. **Puget Sound, Washington**: This area, including places like Seattle and the surrounding waters, offers additional opportunities to see orcas, particularly the Southern Resident killer whale population.\n\n3. **Olympic National Park, Washington**: The coastal areas of the park provide a stunning backdrop for spotting orcas, especially during their migration periods.\n\nWhen planning a trip for whale watching, consider peak seasons for orca activity and book tours with reputable operators who adhere to re

In [4]:
result.response_metadata

{'token_usage': {'completion_tokens': 228,
  'prompt_tokens': 67,
  'total_tokens': 295,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_name': 'gpt-4o-2024-08-06',
 'system_fingerprint': 'fp_50cad350e4',
 'finish_reason': 'stop',
 'logprobs': None}

## Tools

每當你想讓模型與外部系統互動時，tool 就非常有用。

外部系統（例如 API）往往需要特定的輸入 schema 或 payload，而非自然語言。

舉例來說，當我們把一個 API 綁定為 tool 時，就等於讓模型認識到所需的輸入 schema。

模型會根據使用者的自然語言輸入，選擇是否呼叫某個 tool。

而且，它會回傳一個符合該 tool schema 的輸出。

[許多 LLM provider 都支援 tool calling](https://docs.langchain.com/oss/python/integrations/chat)，而 LangChain 中的 [tool calling 介面](https://blog.langchain.com/improving-core-tool-interfaces-and-docs-in-langchain/)非常簡單。

你只要把任何 Python `function` 傳入 `ChatModel.bind_tools(function)` 即可。

![Screenshot 2024-08-19 at 7.46.28 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbab08dc1c17a7a57f9960_chain2.png)

讓我們示範一個簡單的 tool calling 範例！

`multiply` 函式就是我們的 tool。

In [5]:
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

llm_with_tools = llm.bind_tools([multiply])

如果我們傳入一段輸入（例如 `"What is 2 multiplied by 3"`），就會看到回傳了一個 tool call。

這個 tool call 帶有符合我們函式輸入 schema 的特定參數，以及要呼叫的函式名稱。

```
{'arguments': '{"a":2,"b":3}', 'name': 'multiply'}
```

In [8]:
tool_call = llm_with_tools.invoke([HumanMessage(content=f"What is 2 multiplied by 3", name="Lance")])

In [9]:
tool_call.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': 'call_lBBBNo5oYpHGRqwxNaNRbsiT',
  'type': 'tool_call'}]

## 把 messages 當作 state 使用

有了這些基礎，我們現在可以在 graph state 中使用 [messages](https://docs.langchain.com/oss/python/langchain/overview#messages) 了。

讓我們把 state（命名為 `MessagesState`）定義成一個只有單一 key `messages` 的 `TypedDict`。

`messages` 就只是一個 message 的串列，如同我們上面所定義的（例如 `HumanMessage` 等）。

In [1]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage

class MessagesState(TypedDict):
    messages: list[AnyMessage]

## Reducers

現在，我們遇到了一個小問題！

如同前面討論的，每個 node 都會為我們的 state key `messages` 回傳一個新值。

但是，這個新值會覆蓋掉先前的 `messages` 值！

隨著 graph 執行，我們希望把 message **附加（append）**到 `messages` 這個 state key 上。

我們可以用 [reducer 函式](https://docs.langchain.com/oss/python/langgraph/graph-api#reducers)來解決這個問題。

Reducer 用來指定 state 更新該如何進行。

如果沒有指定 reducer 函式，就會假設對該 key 的更新應該*覆蓋*它，就像我們之前看到的那樣。

但是，如果要附加 message，我們可以使用內建的 `add_messages` reducer。

這能確保任何新的 message 都會被附加到既有的 message 串列上。

我們只需要用 `add_messages` reducer 函式作為 metadata，來標註（annotate）`messages` 這個 key 即可。

In [10]:
from typing import Annotated
from langgraph.graph.message import add_messages

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

由於在 graph state 中放一個 message 串列實在太常見了，LangGraph 提供了一個內建的 [`MessagesState`](https://docs.langchain.com/oss/python/langgraph/graph-api#messagesstate)！

`MessagesState` 的定義如下：

* 內建單一個 `messages` key
* 它是一個 `AnyMessage` 物件的串列
* 它使用 `add_messages` reducer

我們通常會使用 `MessagesState`，因為相較於上面那樣自訂 `TypedDict`，它更精簡。

In [13]:
from langgraph.graph import MessagesState

class MessagesState(MessagesState):
    # Add any keys needed beyond messages, which is pre-built 
    pass

為了再深入一點，我們可以單獨看看 `add_messages` reducer 是怎麼運作的。

In [ ]:
# 初始 state
initial_messages = [AIMessage(content="Hello! How can I assist you?", name="Model"),
                    HumanMessage(content="I'm looking for information on marine biology.", name="Lance")
                   ]

# 要加入的新 message
new_message = AIMessage(content="Sure, I can help with that. What specifically are you interested in?", name="Model")

# 測試
add_messages(initial_messages , new_message)

## 我們的 graph

現在，讓我們把 `MessagesState` 搭配一個 graph 來使用。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
    
# Node
def tool_calling_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# 建構 graph
builder = StateGraph(MessagesState)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_edge(START, "tool_calling_llm")
builder.add_edge("tool_calling_llm", END)
graph = builder.compile()

# 檢視
display(Image(graph.get_graph().draw_mermaid_png()))

如果我們傳入 `Hello!`，LLM 會直接回覆，不帶任何 tool call。

In [15]:
messages = graph.invoke({"messages": HumanMessage(content="Hello!")})
for m in messages['messages']:
    m.pretty_print()

================================ Human Message =================================

Hello!
================================== Ai Message ==================================

Hi there! How can I assist you today?


當 LLM 判斷輸入或任務需要某個 tool 所提供的功能時，它就會選擇使用該 tool。

In [16]:
messages = graph.invoke({"messages": HumanMessage(content="Multiply 2 and 3")})
for m in messages['messages']:
    m.pretty_print()

================================ Human Message =================================

Multiply 2 and 3!
================================== Ai Message ==================================
Tool Calls:
  multiply (call_Er4gChFoSGzU7lsuaGzfSGTQ)
 Call ID: call_Er4gChFoSGzU7lsuaGzfSGTQ
  Args:
    a: 2
    b: 3
